In [ ]:
import os,glob,re
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import math
from shapely import wkt
import geopandas as gpd
from shapely.ops import unary_union
from lxml import etree
from pykml.factory import KML_ElementMaker as KML
import folium
import folium.plugins
import unicodedata
import dimsim
from datetime import datetime, timedelta
import openpyxl

In [2]:
def extractYZPZ(df):
    '''
    提取养殖品种
    '''
    print("check df.index:")
    print(f"总长{len(df)},最大索引{df.index.max()}")
    yzxx = df['养殖品种/预计亩产量'].str.replace('斤/亩','')
    yzxx = yzxx.str.split('，',expand=True)
    yzxx = yzxx.fillna('/')
    yzpz = pd.DataFrame(columns=yzxx.columns,index=yzxx.index)
    mcl = pd.DataFrame(columns=yzxx.columns,index=yzxx.index)
    print(f"养殖品种、亩产量拆分")
    for c in yzxx.columns:
        idx = yzxx[c]!='/'
        yzpz.loc[idx,c] = yzxx.loc[idx,c].str.split(':',expand=True)[0]
        mcl.loc[idx,c] = yzxx.loc[idx,c].str.split(':',expand=True)[1]
    
    yzpz = yzpz.fillna('/')
    mcl = mcl.fillna(0)
    mcl = mcl.to_numpy().astype('float')
    yzpz_unq = np.unique(yzpz.to_numpy())
    n = yzpz_unq.shape[0]-1
    print(f"共{n}个品种")
    for i,pz in enumerate(yzpz_unq[yzpz_unq!='/']):
        print(f"{i+1}/{n}:{pz}")
        pz_idx = np.argwhere(yzpz==pz)
        print(pz_idx)
        df.loc[df.index[pz_idx[:,0]],f'{pz}亩产量(斤/亩)'] = mcl[yzpz==pz]

    df['总产量(斤/亩)']=df[df.columns[0-n:]].sum(axis=1)
    df['养殖品种数量']=(df[df.columns[-1-n:-1]] >= 0).sum(axis=1)
    return df,yzpz_unq

In [3]:
# ===== 构建匹配键 =====
def make_key(df):
    return df[["养殖经营人名称", "联系方式", "身份证号", "统一社会信用代码", "地址"]].astype(str).agg("-".join, axis=1)

### 1、统计所有主体数量和清单

In [3]:
# rawpath = r'E:\江苏省养殖池塘上图入库项目\填报数据统计\13个县4条鱼主体统计\丹阳4月9日'
# os.chdir(rawpath)
# # df_mcl=pd.read_excel('高邮市0704.xlsx')
# df_mcl=pd.read_excel('丹阳市.xlsx')#,skiprows=1

In [4]:
# df=df_mcl.copy()

In [5]:
# # 按指定字段划分主体并统计数据
# result = df_mcl.groupby(['养殖经营人名称','身份证号','统一社会信用代码','联系方式','地址']).agg({
#     "养殖经营人名称": "first",
#     "联系方式": "first",
#     "身份证号": "first",
#     "统一社会信用代码": "first",
#     "地址": "first",
#     '图斑编号':list
#     })

In [6]:
# # 获取地址信息，并定义统计层级
# address=result['地址'].str.split('-',expand=True)
# address['区镇']=address[2]+'-'+address[3]
# address_unq=np.unique(address['区镇'].tolist())
# address_unq

In [7]:
# # 创建统计表
# row_index=address_unq
# column_index=['合计']
# result_tj = pd.DataFrame(index=row_index, columns=column_index)

In [8]:
# # 筛选统计数据
# for j in address_unq:
#     for i in ['合计']:
#         if i == '合计':
#             idx3=result['地址'].str.contains(j)
#             result_tj.loc[j,i]=len(result[idx3])

In [9]:
# # 主体清单导出
# result_tj.to_excel(os.path.join("丹阳各镇总主体数量test.xlsx"))
# result.to_excel(os.path.join("丹阳主体_总test.xlsx"))

### 2、统计特定条件主体数量

In [854]:
rawpath = r'E:\江苏省养殖池塘上图入库项目\填报数据统计\13个县4条鱼主体统计\7月新\0717建湖'
os.chdir(rawpath)
df_mcl=pd.read_excel('建湖0717.xlsx')
# df_mcl=pd.read_excel('金湖县0409.xlsx')#,skiprows=1

In [855]:
idx2=df_mcl['图斑面积']!='/'
idx22=df_mcl['图斑面积']=='/'
df_mcl['面积_亩']=''
df_mcl.loc[idx2,'面积_亩']=df_mcl.loc[idx2,'图斑面积'].astype(float)*0.0015
# df_mcl.loc[idx22,'面积_亩']=0.001

In [856]:
# 提取成品养殖、有面积的数据
# idx1=df_mcl['用途'].str.contains('成品养殖')|df_mcl['用途'].str.contains('苗种培育')
idx1=df_mcl['用途'].str.contains('成品养殖')
idx2=df_mcl['图斑面积']!='/'
idx=idx1&idx2
df=df_mcl[idx].copy()

In [104]:
# idx2=df_mcl['图斑面积']!='/'
# df=df_mcl[idx2].copy()

In [857]:
# 按指定字段划分原始信息表中主体并统计数据
df_z=df_mcl.copy()
result_z = df_z.groupby(['养殖经营人名称','身份证号','统一社会信用代码','联系方式','地址']).agg({
    "养殖经营人名称": "first",
    "联系方式": "first",
    "身份证号": "first",
    "统一社会信用代码": "first",
    "地址": "first",
    '图斑编号':list,
    })

In [595]:
# 按指定字段划分成品养殖且有面积数据的主体并统计数据
result_z = df.groupby(['养殖经营人名称','身份证号','统一社会信用代码','联系方式','地址']).agg({
    "养殖经营人名称": "first",
    "联系方式": "first",
    "身份证号": "first",
    "统一社会信用代码": "first",
    "地址": "first",
    '图斑编号':list,
    '面积_亩':"sum"
    })

result_z.to_excel(os.path.join("无锡常州鳊鱼主体0710_总.xlsx"))

In [858]:
# 区分淡海水品种
npz = ['其他种类','南美白对虾','螺','鲈鱼']

for p in npz:
        idx0 = df['养殖品种/预计亩产量'].str.contains(p)
        if len(df.loc[idx0,'水体类型'])>0:
            stlx = df.loc[idx0,'水体类型'].unique()
            for s in stlx[stlx!='/']:
                idx = (df['水体类型']==s) & (idx0)
                df.loc[idx,'养殖品种/预计亩产量'] = df.loc[idx,'养殖品种/预计亩产量'].str.replace(p,f"{s}{p}")

In [859]:
# 提取各品种亩产量
df,yzpz_unq= extractYZPZ(df)

check df.index:
总长4582,最大索引4838
养殖品种、亩产量拆分
共20个品种
1/20:乌鳢
[[  86    0]
 [1254    0]
 [1320    0]
 [1560    0]
 [1561    0]
 [1562    0]
 [1563    0]
 [1564    0]
 [1631    0]
 [1632    0]
 [1633    0]
 [1634    0]
 [1635    0]
 [1636    0]
 [1645    0]
 [1646    0]
 [1649    0]
 [1988    1]
 [1989    1]
 [1990    1]
 [1991    1]
 [1992    1]
 [2276    0]
 [2277    0]
 [2278    0]
 [2279    0]
 [2280    0]
 [2281    0]
 [2282    0]
 [2447    1]
 [2448    1]
 [2543    2]
 [2544    2]
 [2545    2]
 [2546    2]
 [2547    2]
 [2548    2]
 [2549    2]
 [2550    2]
 [3333    1]
 [3334    0]
 [3335    0]
 [3420    0]
 [3421    0]
 [3422    0]
 [3423    0]
 [3424    0]
 [3425    0]
 [3426    0]
 [3427    0]
 [3428    0]
 [3429    0]
 [3430    0]
 [3748    0]
 [3929    0]
 [4533    0]
 [4534    0]
 [4535    0]
 [4536    0]
 [4537    0]
 [4538    0]
 [4539    0]
 [4540    0]
 [4541    0]
 [4542    0]
 [4551    0]
 [4552    0]
 [4553    0]]
2/20:克氏原螯虾
[[ 113    1]
 [ 124    0]
 [ 125    0]
 [ 126 

In [860]:
# 计算指定品种总产量
pzlist=['鳊鲂','鲫鱼','淡水鲈鱼','泥鳅','黄鳝','蛙','乌鳢']
# pzlist=['黄鳝','蛙','乌鳢']
for pz in pzlist:
    if pz+'亩产量(斤/亩)' in df.columns:
        idx1=df[pz+'亩产量(斤/亩)']>=0
        df.loc[idx1,pz+'产量']=df.loc[idx1,pz+'亩产量(斤/亩)']*df.loc[idx1,'面积_亩']

In [861]:
df_tj=df.copy()

In [84]:
# 按指定字段划分主体并统计数据
result = df_tj.groupby(['养殖经营人名称','身份证号','统一社会信用代码','联系方式','地址']).agg({
    "养殖经营人名称": "first",  # 或 "max"/"min"（如果身份证号不同，需去重逻辑）
    "联系方式": "first",
    "身份证号": "first",
    "统一社会信用代码": "first",
    "地址": "first",
    "面积_亩": "sum",
    "鳊鲂产量": "sum",
    "淡水鲈鱼产量": "sum",
    "鲫鱼产量": "sum",
    "泥鳅产量": "sum",
    "图斑编号":list
    })

In [863]:
# 按指定字段划分主体并统计数据-多个品种
result = df_tj.groupby(['养殖经营人名称','身份证号','统一社会信用代码','联系方式','地址']).agg({
    "养殖经营人名称": "first",  # 或 "max"/"min"（如果身份证号不同，需去重逻辑）
    "联系方式": "first",
    "身份证号": "first",
    "统一社会信用代码": "first",
    "地址": "first",
    "面积_亩": "sum",
    "鳊鲂产量": "sum",
    "鲫鱼产量": "sum",
    "乌鳢产量": "sum",
    "蛙产量": "sum",
    "图斑编号":list
    })

In [864]:
result['养殖品种']=''

In [865]:
for pz in pzlist:
    if pz+'产量' in result.columns:
        idx1=result[pz+'产量']>0
        result.loc[idx1,'养殖品种']+=pz+';'

In [720]:
# 大于5亩、主体清单
idx1=result['面积_亩']>=5
idx2=result['养殖品种'].str.contains(';')
idx=idx1&idx2
result1=result[idx].copy()

In [866]:
# 不筛面积主体清单
idx2=result['养殖品种'].str.contains(';')
result2=result[idx2].copy()
result2['所属乡镇']=result2['地址'].str.split('-',expand=True)[3]
len(result2)

127

In [823]:
# 不筛面积除指定品种外，其他主体清单
idx3=~result['养殖品种'].str.contains(';')
result3=result[idx3].copy()

In [576]:
# result1['所属乡镇']=result1['地址'].str.split('-',expand=True)[3]
# result2['所属乡镇']=result2['地址'].str.split('-',expand=True)[3]
# result['所属乡镇']=result['地址'].str.split('-',expand=True)[3]
# result_z['所属乡镇']=result_z['地址'].str.split('-',expand=True)[3]

In [58]:
# 主体清单导出
result2.to_excel(os.path.join("溧阳成品养殖、苗种培育七条鱼主体0718.xlsx"))

In [446]:
# 主体清单导出
result1.to_excel(os.path.join("金湖县5亩以上成品养殖四条鱼主体0707.xlsx"))
# result2.to_excel(os.path.join("金湖县不筛面积成品养殖四条鱼主体0707.xlsx"))
# 主体清单导出
# result.to_excel(os.path.join("金湖县成品养殖主体0707.xlsx"))
# result_z.to_excel(os.path.join("金湖县主体0707_总.xlsx"))

In [690]:
# 获取地址信息，并定义统计层级
address=result['地址'].str.split('-',expand=True)
address['区镇']=address[2]+'-'+address[3]
address_unq=np.unique(address[2].tolist())
# address_unq=np.unique(address['区镇'].tolist())
address_unq

array(['洪泽区', '涟水县', '淮安工业园区', '淮安经济技术开发区', '淮阴区', '清江浦区', '盱眙县', '金湖县'],
      dtype='<U9')

In [90]:
# 创建统计表
row_index=address_unq
column_index=['鳊鲂','鲫鱼','淡水鲈鱼','泥鳅','四鱼合计']
result_tj = pd.DataFrame(index=row_index, columns=column_index)

In [91]:
# 筛选统计数据
# 创建统计表
for j in address_unq:
    for i in ['鳊鲂','鲫鱼','淡水鲈鱼','泥鳅','四鱼合计']:
        if i=='四鱼合计':
            idx=result['养殖品种'].str.contains('鳊鲂')|result['养殖品种'].str.contains('鲫鱼')|result['养殖品种'].str.contains('淡水鲈鱼')|result['养殖品种'].str.contains('泥鳅')
            idx2=result['面积_亩']>=5
            idx3=result['地址'].str.contains(j)
            result_tj.loc[j,i]=len(result[idx&idx2&idx3])
        elif i in ['鳊鲂','鲫鱼','淡水鲈鱼','泥鳅']:
            idx=result['养殖品种'].str.contains(i)
            idx2=result['面积_亩']>=5
            idx3=result['地址'].str.contains(j)
            result_tj.loc[j,i]=len(result[idx&idx2&idx3])

In [579]:
# 筛选统计数据-金湖
# 创建统计表
row_index=address_unq
column_index=['鳊鲂','鲫鱼','淡水鲈鱼','泥鳅','四鱼合计','黄鳝','蛙','合计']
result_tj = pd.DataFrame(index=row_index, columns=column_index)

for j in address_unq:
    for i in ['鳊鲂','鲫鱼','淡水鲈鱼','泥鳅','四鱼合计','黄鳝','蛙','合计']:
        if i == '合计':
            idx=result['养殖品种'].str.contains(';')
            idx2=result['面积_亩']>=5
            idx3=result['地址'].str.contains(j)
            result_tj.loc[j,i]=len(result[idx&idx2&idx3])
        elif i in ['鳊鲂','鲫鱼','淡水鲈鱼','泥鳅','黄鳝','蛙']:
            idx=result['养殖品种'].str.contains(i)
            idx2=result['面积_亩']>=5
            idx3=result['地址'].str.contains(j)
            result_tj.loc[j,i]=len(result[idx&idx2&idx3])
        elif i=='四鱼合计':
            idx=result['养殖品种'].str.contains('鳊鲂')|result['养殖品种'].str.contains('鲫鱼')|result['养殖品种'].str.contains('淡水鲈鱼')|result['养殖品种'].str.contains('泥鳅')
            idx2=result['面积_亩']>=5
            idx3=result['地址'].str.contains(j)
            result_tj.loc[j,i]=len(result[idx&idx2&idx3])

In [363]:
# 创建统计表
row_index=address_unq
column_index=['鳊鲂','鲫鱼','淡水鲈鱼','泥鳅','四鱼合计','非四鱼合计','所有主体','不筛面积四鱼合计']
result_tj = pd.DataFrame(index=row_index, columns=column_index)

In [364]:
# 筛选统计数据
for j in address_unq:
    for i in ['鳊鲂','鲫鱼','淡水鲈鱼','泥鳅','四鱼合计','非四鱼合计','所有主体','不筛面积四鱼合计']:
        if i == '四鱼合计':
            idx=result['养殖品种'].str.contains(';')
            idx2=result['面积_亩']>=5
            idx3=result['地址'].str.contains(j)
            result_tj.loc[j,i]=len(result[idx&idx2&idx3])
        elif i in ['鳊鲂','鲫鱼','淡水鲈鱼','泥鳅']:
            idx=result['养殖品种'].str.contains(i)
            idx2=result['面积_亩']>=5
            idx3=result['地址'].str.contains(j)
            result_tj.loc[j,i]=len(result[idx&idx2&idx3])
        elif i == '非四鱼合计':
            idx=~result['养殖品种'].str.contains(';')
            idx2=result['面积_亩']>=5
            idx3=result['地址'].str.contains(j)
            result_tj.loc[j,i]=len(result[idx&idx2&idx3])
        elif i == '所有主体':
            idx3=result_z['地址'].str.contains(j)
            result_tj.loc[j,i]=len(result_z[idx3])
        if i == '不筛面积四鱼合计':
            idx=result['养殖品种'].str.contains(';')
#             idx2=result['面积_亩']>=5
            idx3=result['地址'].str.contains(j)
            result_tj.loc[j,i]=len(result[idx&idx3])

In [824]:
# 创建统计表
row_index=address_unq
column_index=['鳊鲂','鲫鱼','淡水鲈鱼','泥鳅','黄鳝','蛙','乌鳢','合计']
result_tj = pd.DataFrame(index=row_index, columns=column_index)
# 筛选统计数据
# 创建统计表
for j in address_unq:
    for i in ['鳊鲂','鲫鱼','淡水鲈鱼','泥鳅','黄鳝','蛙','乌鳢','合计']:
        if i=='合计':
            idx=result['养殖品种'].str.contains(';')
            idx3=result['地址'].str.contains(j)
            result_tj.loc[j,i]=len(result[idx&idx3])
        elif i in ['鳊鲂','鲫鱼','淡水鲈鱼','泥鳅','黄鳝','蛙','乌鳢']:
            idx=result['养殖品种'].str.contains(i)
            idx3=result['地址'].str.contains(j)
            result_tj.loc[j,i]=len(result[idx&idx3])

In [294]:
# 创建统计表
row_index=address_unq
column_index=['黄鳝','蛙','乌鳢','合计']
result_tj = pd.DataFrame(index=row_index, columns=column_index)
# 筛选统计数据
# 创建统计表
for j in address_unq:
    for i in ['黄鳝','蛙','乌鳢','合计']:
        if i=='合计':
            idx=result['养殖品种'].str.contains(';')
            idx3=result['地址'].str.contains(j)
            result_tj.loc[j,i]=len(result[idx&idx3])
        elif i in ['黄鳝','蛙','乌鳢']:
            idx=result['养殖品种'].str.contains(i)
            idx3=result['地址'].str.contains(j)
            result_tj.loc[j,i]=len(result[idx&idx3])

In [692]:
# 统计数据导出
result_tj.to_excel(os.path.join("淮安市七鱼主体数量0721.xlsx"))

In [693]:
result_tj

,鳊鲂,鲫鱼,淡水鲈鱼,泥鳅,黄鳝,蛙,乌鳢,合计
洪泽区,17,73,1,0,0,0,0,78
涟水县,0,532,0,1,0,0,0,533
淮安工业园区,0,0,0,0,0,0,0,0
淮安经济技术开发区,0,2,0,0,0,0,0,2
淮阴区,20,205,0,4,0,0,0,212
清江浦区,1,36,1,0,0,0,0,36
盱眙县,0,0,0,0,0,0,0,0
金湖县,680,700,4,1,1,1,0,725


### 匹配信息表并导出

In [867]:
# 原始数据匹配
# ===== 参数配置 =====
output_path = "七鱼清单.xlsx"  # 输出文件路径
output_path2 = "其他清单.xlsx"  
output_path3 = "异常数据.xlsx"# 输出文件路径
df_b = result2.copy() # 匹配不同结果表格
# df_c = result.copy()# 相同用途其他
df_c = result_z.copy() # 其他所有

In [847]:
# ===== 关键词列表 =====其他所有
keywords = ["鲫鱼", "鳊鲂", "泥鳅", "鲈鱼", "黄鳝", "蛙","乌鳢"] #, "黄鳝", "蛙"
# ===== 读取数据 =====
# idx1=df_mcl['用途'].str.contains('成品养殖')|df_mcl['用途'].str.contains('苗种培育')
idx1=df_mcl['用途'].str.contains('成品养殖')
df_a = df_mcl[idx1].copy()
# df_a2 = df_mcl.copy()
# ===== 筛选 A 表中品种字段包含关键词的数据 =====
mask = df_a["养殖品种/预计亩产量"].astype(str).apply(lambda x: any(k in x for k in keywords))
filtered_a = df_a[mask].copy()

# mask2 = df_a2["养殖品种/预计亩产量"].astype(str).apply(lambda x: not any(k in x for k in keywords))
# filtered_c = df_a2[mask2].copy()
common_index = df_mcl.index.intersection(filtered_a.index)
filtered_c = df_mcl.drop(common_index)

In [417]:
# ===== 关键词列表 =====其他所有2
keywords = ["鲫鱼", "鳊鲂", "泥鳅", "鲈鱼", "黄鳝", "蛙","乌鳢"] #, "黄鳝", "蛙"
# ===== 读取数据 =====
idx1=df_mcl['用途'].str.contains('成品养殖')|df_mcl['用途'].str.contains('苗种培育')
# idx1=df_mcl['用途'].str.contains('成品养殖')
df_a = df_mcl[idx1].copy()
# df_a2 = df_mcl.copy()
# ===== 筛选 A 表中品种字段包含关键词的数据 =====
mask = df_a["养殖品种/预计亩产量"].astype(str).apply(lambda x: any(k in x for k in keywords))
filtered_a = df_a[mask].copy()
idx2 = filtered_a['用途'].str.contains('成品养殖')
idx3 = filtered_a['养殖品种/预计亩产量'].str.contains('鲫鱼')|filtered_a['养殖品种/预计亩产量'].str.contains('鳊鲂')|filtered_a['养殖品种/预计亩产量'].str.contains('鲈鱼')|filtered_a['养殖品种/预计亩产量'].str.contains('泥鳅')
idx4 = ~filtered_a['养殖品种/预计亩产量'].str.contains('黄鳝')
idx5 = ~filtered_a['养殖品种/预计亩产量'].str.contains('蛙')
idx6 = ~filtered_a['养殖品种/预计亩产量'].str.contains('乌鳢')
drop_a = filtered_a[idx2&idx3&idx4&idx5&idx6]
filtered_a = filtered_a.drop(drop_a.index)
# mask2 = df_a2["养殖品种/预计亩产量"].astype(str).apply(lambda x: not any(k in x for k in keywords))
# filtered_c = df_a2[mask2].copy()
common_index = df_mcl.index.intersection(filtered_a.index)
filtered_c = df_mcl.drop(common_index)

In [868]:
# ===== 关键词列表 =====相同用途其他
keywords = ["鲫鱼", "鳊鲂", "泥鳅", "鲈鱼", "黄鳝", "蛙","乌鳢"] #, "黄鳝", "蛙"
# ===== 读取数据 =====
idx1=df_mcl['用途'].str.contains('成品养殖')
df_a = df_mcl[idx1].copy()
# ===== 筛选 A 表中品种字段包含关键词的数据 =====
mask = df_a["养殖品种/预计亩产量"].astype(str).apply(lambda x: any(k in x for k in keywords))
filtered_a = df_a[mask].copy()

# mask2 = df_a["养殖品种/预计亩产量"].astype(str).apply(lambda x: not any(k in x for k in keywords))
# filtered_c = df_a[mask2].copy()
common_index = df_a.index.intersection(filtered_a.index)
filtered_c = df_a.drop(common_index)

In [848]:
len(filtered_a)+len(filtered_c)

12127

In [852]:
len(matched_a)+len(matched_c)

12121

In [674]:
len(matched_c)

7754

In [872]:
matched_a2["养殖品种/预计亩产量"]

253    草鱼:0.00斤/亩，鲢鱼:0.00斤/亩，鲫鱼:0.00斤/亩
254    草鱼:0.00斤/亩，鲢鱼:0.00斤/亩，鲫鱼:0.00斤/亩
255    草鱼:0.00斤/亩，鲢鱼:0.00斤/亩，鲫鱼:0.00斤/亩
615                          鲫鱼:0.00斤/亩
Name: 养殖品种/预计亩产量, dtype: object

In [869]:
matched_a,matched_a2=pipei(filtered_a,df_b,lx='七鱼')
matched_a.to_excel(output_path)
matched_a2["所在乡镇"] = matched_a2["地址"].astype(str).str.split("-").str[3]
matched_a2 = matched_a2[['养殖经营人名称', '身份证号', '统一社会信用代码', '养殖主体类型', '地址', '所在乡镇', '联系人', '联系方式',
                                   '养殖品种/预计亩产量', '图斑编号', '面积_亩','池塘所有权','池塘所有权人名称','池塘所有权人证件号码','用途']]
matched_a2.to_excel(output_path3)

In [829]:
common_index = df_mcl.index.intersection(matched_a.index)
matched_c = df_mcl.drop(common_index)
matched_c["所在乡镇"] = matched_c["地址"].astype(str).str.split("-").str[3]
matched_c = matched_c[['养殖经营人名称', '身份证号', '统一社会信用代码', '养殖主体类型', '地址', '所在乡镇', '联系人', '联系方式',
                                   '养殖品种/预计亩产量', '图斑编号', '面积_亩','池塘所有权','池塘所有权人名称','池塘所有权人证件号码','用途']]

In [871]:
matched_c,matched_c2=pipei(filtered_c,df_c,lx='其他')
# matched_c=pd.concat([matched_c,matched_a2],axis=0)
# matched_c.to_excel(output_path2)

In [174]:
def pipei(filtered,df,lx):
    filtered["match_key"] = make_key(filtered)
    df["match_key"] = make_key(df)
    # ===== 匹配：从 A 中选出匹配 B 的数据 =====
    matched_keys = set(df["match_key"])
    matched_a = filtered[filtered["match_key"].isin(matched_keys)].copy()
    matched_a2 = filtered[~filtered["match_key"].isin(matched_keys)].copy()
    # ===== 提取“乡镇”信息 =====
    matched_a["所在乡镇"] = matched_a["地址"].astype(str).str.split("-").str[3]
    if lx=='七鱼':
        matched_a["变更理由"]=''
        matched_a.drop(columns=["match_key"], inplace=True)
        matched_a = matched_a[['养殖经营人名称', '身份证号', '统一社会信用代码', '养殖主体类型', '地址', '所在乡镇', '联系人', '联系方式',
                                   '养殖品种/预计亩产量', '图斑编号', '面积_亩','池塘所有权','池塘所有权人名称','池塘所有权人证件号码','用途','变更理由']]
    else:
        matched_a.drop(columns=["match_key"], inplace=True)
        matched_a = matched_a[['养殖经营人名称', '身份证号', '统一社会信用代码', '养殖主体类型', '地址', '所在乡镇', '联系人', '联系方式',
                                   '养殖品种/预计亩产量', '图斑编号', '面积_亩','池塘所有权','池塘所有权人名称','池塘所有权人证件号码','用途']]

    return matched_a,matched_a2

In [430]:
# 按指定字段划分主体并统计数据-多个品种
result2_2 = matched_a.groupby(['养殖经营人名称','身份证号','统一社会信用代码','联系方式','地址']).agg({
    "养殖经营人名称": "first",  # 或 "max"/"min"（如果身份证号不同，需去重逻辑）
    "联系方式": "first",
    "身份证号": "first",
    "统一社会信用代码": "first",
    "地址": "first",
    "图斑编号":list,
    '养殖品种/预计亩产量':list
    })

In [431]:
result2_2['养殖品种']=result2_2['养殖品种/预计亩产量'].apply(lambda x: ','.join(map(str, x)))

In [432]:
# 创建统计表
row_index=address_unq
column_index=['鳊鲂','鲫鱼','鲈鱼','泥鳅','黄鳝','蛙','乌鳢','合计']
result_tj2 = pd.DataFrame(index=row_index, columns=column_index)
# 筛选统计数据
# 创建统计表
for j in address_unq:
    for i in ['鳊鲂','鲫鱼','鲈鱼','泥鳅','黄鳝','蛙','乌鳢','合计']:
        if i=='合计':
            idx3=result2_2['地址'].str.contains(j)
            result_tj2.loc[j,i]=len(result2_2[idx3])
        elif i in ['鳊鲂','鲫鱼','鲈鱼','泥鳅','黄鳝','蛙','乌鳢']:
            idx=result2_2['养殖品种'].str.contains(i)
            idx3=result2_2['地址'].str.contains(j)
            result_tj2.loc[j,i]=len(result2_2[idx&idx3])

In [433]:
# 统计数据导出
result_tj2.to_excel(os.path.join("高邮七鱼主体数量0720.xlsx"))
result_tj2

,鳊鲂,鲫鱼,鲈鱼,泥鳅,黄鳝,蛙,乌鳢,合计
三垛镇,0,330,9,4,0,0,4,346
临泽镇,0,2,0,0,0,0,1,3
卸甲镇,0,9,1,0,0,0,6,15
周山镇,0,2,0,0,0,0,0,2
城南经济新区（车逻镇）,1,2,0,0,0,0,4,5
汤庄镇,0,9,0,0,0,0,3,11
甘垛镇,0,90,4,8,0,0,0,102
界首镇,0,0,0,0,0,0,0,0
经济开发区（马棚街道）,1,5,0,0,0,0,0,5
菱塘回族乡,0,141,0,0,0,0,0,141


In [67]:
# filtered_a["match_key"] = make_key(filtered_a)
# df_b["match_key"] = make_key(df_b)
# # ===== 匹配：从 A 中选出匹配 B 的数据 =====
# matched_keys = set(df_b["match_key"])
# matched_a = filtered_a[filtered_a["match_key"].isin(matched_keys)].copy()
# # ===== 提取“乡镇”信息 =====
# matched_a["所在乡镇"] = matched_a["地址"].astype(str).str.split("-").str[3]
# matched_a["变更理由"]=''
# # ===== 删除辅助列并保存结果 =====
# # matched_a.drop(columns=["match_key"], inplace=True)
# matched_a = matched_a[['养殖经营人名称', '身份证号', '统一社会信用代码', '养殖主体类型', '地址', '所在乡镇', '联系人', '联系方式',
#                            '养殖品种/预计亩产量', '图斑编号', '面积_亩','池塘所有权','池塘所有权人名称','池塘所有权人证件号码','用途','变更理由']]
# # matched_a=matched_a.set_index(["养殖经营人名称", "联系方式", "身份证号", "统一社会信用代码", "地址"])
# matched_a.to_excel(output_path)

In [69]:
# filtered_c["match_key"] = make_key(filtered_c)
# df_c["match_key"] = make_key(df_c)
# # ===== 匹配：从 A 中选出匹配 B 的数据 =====
# matched_keys2 = set(df_c["match_key"])
# matched_c = filtered_c[filtered_c["match_key"].isin(matched_keys2)].copy()
# # ===== 提取“乡镇”信息 =====
# matched_c["所在乡镇"] = matched_c["地址"].astype(str).str.split("-").str[3]
# # ===== 删除辅助列并保存结果 =====
# # matched_a.drop(columns=["match_key"], inplace=True)
# matched_c = matched_c[['养殖经营人名称', '身份证号', '统一社会信用代码', '养殖主体类型', '地址', '所在乡镇', '联系人', '联系方式',
#                            '养殖品种/预计亩产量', '图斑编号', '面积_亩','池塘所有权','池塘所有权人名称','池塘所有权人证件号码','用途']]
# # matched_a=matched_a.set_index(["养殖经营人名称", "联系方式", "身份证号", "统一社会信用代码", "地址"])
# matched_c.to_excel(output_path2)

### 按地址分sheet导出

In [703]:
address=matched_a['地址'].str.split('-',expand=True)
address_unq=np.unique(address[2].tolist())
address_unq1=address_unq[address_unq!='其他区域']
idx1=address[3].str.contains('其他区域')
address2=address[idx1].copy()
address_unq2=np.unique(address2[4].tolist())

address3=matched_c['地址'].str.split('-',expand=True)
address_unq3=np.unique(address3[2].tolist())

address_unq

array(['洪泽区', '涟水县', '淮安经济技术开发区', '淮阴区', '清江浦区', '金湖县'], dtype='<U9')

In [704]:
address_unq_z=list(set(list(address_unq) + list(address_unq3)))
address_unq_z

['涟水县', '金湖县', '淮阴区', '淮安生态文旅区', '淮安经济技术开发区', '清江浦区', '盱眙县', '淮安工业园区', '洪泽区']

In [146]:
with pd.ExcelWriter(output_path.replace('.xlsx','_按乡镇.xlsx')) as writer:
    for i in address_unq1:
        idx=matched_a['地址'].str.contains(i)
        matched_b=matched_a[idx].copy()
        matched_b.to_excel(writer, sheet_name=i, index=False)
    for j in address_unq2:
        idx=matched_a['地址'].str.contains(j)
        matched_b=matched_a[idx].copy()
        matched_b.to_excel(writer, sheet_name=j, index=False)

### 按地址分sheet导出-常规

In [ ]:
with pd.ExcelWriter(output_path) as writer:
    for i in address_unq:
        idx=matched_a['地址'].str.contains(i)
        matched_b=matched_a[idx].copy()
        matched_b.to_excel(writer, sheet_name=i, index=False)

### 按地址分excel导出

In [152]:
for i in address_unq1:
    idx=matched_a['地址'].str.contains(i)
    matched_b=matched_a[idx].copy()
    matched_b.to_excel(i+'.xlsx', index=False)
for j in address_unq2:
    idx=matched_a['地址'].str.contains(j)
    matched_b=matched_a[idx].copy()
    matched_b.to_excel(j+'.xlsx', index=False)

### 按地址分excel导出-常规

In [705]:
e=0
for i in address_unq_z:
    with pd.ExcelWriter(i+'.xlsx') as writer:
        idx=matched_a['地址'].str.contains(i)
        matched_a2=matched_a[idx].copy()
        e=e+len(matched_a2)
        matched_a2.to_excel(writer, sheet_name='七鱼', index=False)
        idx2=matched_c['地址'].str.contains(i)
        matched_c2=matched_c[idx2].copy()
        e=e+len(matched_c2)
        matched_c2.to_excel(writer, sheet_name='其他鱼', index=False)

In [701]:
e=0
for i in address_unq:
    with pd.ExcelWriter(i+'.xlsx') as writer:
        idx=matched_a['地址'].str.contains(i)
        matched_a2=matched_a[idx].copy()
        e=e+len(matched_a2)
        matched_a2.to_excel(writer, sheet_name='七鱼', index=False)
        idx2=matched_c['地址'].str.contains(i)
        matched_c2=matched_c[idx2].copy()
        e=e+len(matched_c2)
        matched_c2.to_excel(writer, sheet_name='其他鱼', index=False)

In [706]:
e

38080

In [42]:
i='新庄街道'
with pd.ExcelWriter(i+'.xlsx') as writer:
    idx=matched_a['地址'].str.contains(i)
    matched_a2=matched_a[idx].copy()
    e=e+len(matched_a2)
    matched_a2.to_excel(writer, sheet_name='七鱼', index=False)
    idx2=matched_c['地址'].str.contains(i)
    matched_c2=matched_c[idx2].copy()
    e=e+len(matched_c2)
    matched_c2.to_excel(writer, sheet_name='其他鱼', index=False)

In [87]:
e

31420

In [451]:
# 创建统计表
row_index=address_unq
column_index=['主体','塘口']
result_tj5 = pd.DataFrame(index=row_index, columns=column_index)

In [454]:
for j in address_unq:
    result_tj5.loc[j,'主体']=len(result2[result2['地址'].str.contains(j).copy()])
    result_tj5.loc[j,'塘口']=len(matched_a[matched_a['地址'].str.contains(j).copy()])

In [456]:
# 统计数据导出
result_tj5.to_excel(os.path.join("响水苗种培育成品养殖主体、池塘数量0717.xlsx"))

In [455]:
result_tj5

,主体,塘口
南河镇,1,1
县开发区,13,22
双港镇,10,23
响水县滩涂开发管理局驻陈港办事处,0,0
大有镇,24,38
小尖镇,28,48
江苏兴海控股集团有限公司,0,0
江苏农垦金鲤渔业有限公司,1,15
江苏银宝盐业有限公司,0,0
省属灌东盐场,22,455


In [427]:
result_a=matched_a.groupby(['养殖经营人名称','身份证号','统一社会信用代码','联系方式','地址']).agg({
    "养殖经营人名称": "first",  # 或 "max"/"min"（如果身份证号不同，需去重逻辑）
    "联系方式": "first",
    "身份证号": "first",
    "统一社会信用代码": "first",
    "地址": "first",
    "所在乡镇":"first",
    "养殖主体类型":"first",
    "鳊鲂产量": "sum",
    "淡水鲈鱼产量": "sum",
    "泥鳅产量": "sum",
    "鲫鱼产量": "sum",
    "蛙产量": "sum",
    "黄鳝产量": "sum",
    "图斑编号":list
    })

In [428]:
result_a['养殖品种']=''
# 根据产量得出养殖品种
if '鳊鲂产量' in result_a.columns:
    idx1=result_a['鳊鲂产量']>0
    result_a.loc[idx1,'养殖品种']+='鳊鲂;'
if '淡水鲈鱼产量' in result_a.columns:
    idx2=result_a['淡水鲈鱼产量']>0
    result_a.loc[idx2,'养殖品种']+='淡水鲈鱼;'
if '鲫鱼产量' in result_a.columns:
    idx3=result_a['鲫鱼产量']>0
    result_a.loc[idx3,'养殖品种']+='鲫鱼;'
if '泥鳅产量' in result_a.columns:
    idx4=result_a['泥鳅产量']>0
    result_a.loc[idx4,'养殖品种']+='泥鳅;'
if '黄鳝产量' in result_a.columns:
    idx5=result_a['黄鳝产量']>0
    result_a.loc[idx5,'养殖品种']+='黄鳝;'
if '蛙产量' in result_a.columns:
    idx6=result_a['蛙产量']>0
    result_a.loc[idx6,'养殖品种']+='蛙;'

In [429]:
result_a.to_excel(output_path.replace('.xlsx','-汇总-原.xlsx'), index=False)

### 匹配所有主体清单

In [156]:
# 原始数据匹配
# ===== 参数配置 =====
output_path = "丹阳市所有养殖主体0409清单.xlsx"  # 输出文件路径

In [198]:
# ===== 读取数据 =====
df_a = df_mcl.copy()
df_b = result_z.copy()

In [199]:
filtered_a = df_a.copy()

In [200]:
filtered_a["match_key"] = make_key(filtered_a)
df_b["match_key"] = make_key(df_b)
# ===== 匹配：从 A 中选出匹配 B 的数据 =====
matched_keys = set(df_b["match_key"])
matched_a = filtered_a[filtered_a["match_key"].isin(matched_keys)].copy()
# ===== 提取“乡镇”信息 =====
matched_a["所在乡镇"] = matched_a["地址"].astype(str).str.split("-").str[3]
# ===== 删除辅助列并保存结果 =====
matched_a.drop(columns=["match_key"], inplace=True)
matched_a = matched_a[['养殖经营人名称', '身份证号', '养殖主体类型','统一社会信用代码', '地址', '所在乡镇', '联系人', '联系方式',
                           '养殖品种/预计亩产量', '图斑编号', '面积_亩']]
matched_a.to_excel(output_path, index=False)

In [874]:
rawpath = r'E:\江苏省养殖池塘上图入库项目\填报数据统计\13个县4条鱼主体统计\7月新\各地产量为0、无图斑数据'
os.chdir(rawpath)
for i in ['宝应县','丹阳市','高邮市','淮安区','建湖县','金湖县','溧阳市','射阳县','吴江区','武进区','响水县','兴化市','宜兴市']:

    df_mcl=pd.read_excel(i+'.xlsx')
    idx2=df_mcl['图斑面积']!='/'
    idx22=df_mcl['图斑面积']=='/'
    df_mcl['面积_亩']=''
    df_mcl.loc[idx2,'面积_亩']=df_mcl.loc[idx2,'图斑面积'].astype(float)*0.0015
    df_mcl.loc[idx22,'面积_亩']='/'
    df_mcl_a = df_mcl[['养殖经营人名称', '身份证号', '统一社会信用代码', '养殖主体类型', '地址', '联系人', '联系方式',
                                   '养殖品种/预计亩产量', '图斑编号', '面积_亩','池塘所有权','池塘所有权人名称','池塘所有权人证件号码','用途']]
    df_mcl_a.to_excel(i+'_异常数据.xlsx', index=False)
# df_mcl=pd.read_excel('金湖县0409.xlsx')#,skiprows=1